Mount Google Drive and initialize project

In [15]:
from google.colab import drive
import os
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Project configuration
PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

# Set working directory
os.chdir(PROJECT_DIR)

print("Project mounted successfully!")
print("Working directory:", os.getcwd())
print("Python version:", sys.version)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project mounted successfully!
Working directory: /content/drive/MyDrive/DumLLM
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


#GitHub Auto Sync

In [ ]:

import os
import subprocess
import base64
from google.colab import userdata

# ==========================================
# DumLLM - GitHub Auto Sync
# ==========================================

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"
REPO_URL = "https://github.com/ShubhamKumarSahu-svg/DumLLM.git"
BRANCH = "main"

os.chdir(PROJECT_DIR)


def git(command, check=True, auth=False):
    cmd = ["git"]

    if auth:
        cmd += ["-c", f"http.extraheader=AUTHORIZATION: basic {AUTH}"]

    cmd += command

    result = subprocess.run(
        cmd,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or "Git command failed!")

    return result


# 1. Verify repository
if not os.path.isdir(".git"):
    raise RuntimeError("Git repository not found!")

# 2. Configure Git identity
git(["config", "user.name", "Shubham Kumar Sahu"])
git(["config", "user.email", "shubhamisok@gmail.com"])

# 3. Configure remote
git(["remote", "set-url", "origin", REPO_URL])

# 4. Ensure important files are ignored
ignore_rules = [
    ".venv/", "venv/", "ENV/", "env/",
    "__pycache__/", "*.py[cod]",
    ".ipynb_checkpoints/", "**/.ipynb_checkpoints/",
    ".env", ".env.*", "!.env.example",

    # Documents and datasets
    "*.pdf", "*.zip", "*.csv",

    # Model weights and checkpoints
    "*.pth", "*.ckpt", "*.safetensors",
    "/gpt2/", "/model.pth",
    "checkpoints/", "/sms_spam_collection/"
]

existing = set()
if os.path.exists(".gitignore"):
    with open(".gitignore", "r") as f:
        existing = set(f.read().splitlines())

new_rules = [rule for rule in ignore_rules if rule not in existing]

if new_rules:
    with open(".gitignore", "a") as f:
        if existing:
            f.write("\n")
        f.write("\n".join(new_rules) + "\n")

# 5. Get GitHub token
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError("GITHUB_TOKEN missing from Colab Secrets!")

AUTH = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

# 6. Verify branch
current_branch = git(["branch", "--show-current"]).stdout.strip()

if current_branch != BRANCH:
    raise RuntimeError(f"Expected branch '{BRANCH}', found '{current_branch}'.")

# 7. Fetch remote changes
print("Checking GitHub...")
git(["fetch", "origin", BRANCH], auth=True)

# 8. Ensure remote is an ancestor of local HEAD
sync = git(
    ["merge-base", "--is-ancestor", f"origin/{BRANCH}", "HEAD"],
    check=False
)

if sync.returncode != 0:
    raise RuntimeError(
        "GitHub contains commits not present locally, or histories differ.\n"
        "Synchronize your branch before committing. No push was attempted."
    )

# 9. Ask for commit message
message = input("Enter commit message: ").strip()

if not message:
    raise ValueError("Commit message cannot be empty!")

# 10. Stage changes
git(["add", "-A"])

# 11. Check for oversized staged files
files = git(["diff", "--cached", "--name-only"]).stdout.splitlines()

for file in files:
    result = subprocess.run(
        ["git", "cat-file", "-s", f":{file}"],
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        size = int(result.stdout.strip())

        if size > 95 * 1024 * 1024:
            git(["reset"])
            raise RuntimeError(
                f"Large file detected: {file} "
                f"({size / 1024**2:.1f} MB).\n"
                "It was not committed. Add it to .gitignore or remove it from tracking."
            )

# 12. Check whether anything changed
changes = git(["diff", "--cached", "--quiet"], check=False)

if changes.returncode == 0:
    print("No changes to commit.")
    exit()

if changes.returncode != 1:
    raise RuntimeError("Could not inspect staged changes.")

# 13. Commit
git(["commit", "-m", message])
print("\nCommit created successfully!")

# 14. Push
print("\nPushing to GitHub...")

result = git(
    ["push", "-u", "origin", f"HEAD:{BRANCH}"],
    check=False,
    auth=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "Push failed. Your local commit is preserved. "
        "Synchronize with GitHub before retrying."
    )

# 15. Success
print("\n" + "=" * 45)
print("DumLLM GitHub Sync Successful!")
print("=" * 45)
print("Commit:", message)
print("Branch:", BRANCH)
print("Repository:", REPO_URL)
print("\nLatest commit:")
print(git(["log", "-1", "--oneline"]).stdout)

Checking GitHub...
